# CMO emitter-aware Wikipedia airborne-radar Neo4j KG

Replacement for the previous `wikipedia_airborne_radars_neo4j_kg.ipynb`. It wipes the selected Neo4j database and rebuilds a reference KG with an ontology aligned to `LuaHistory_2026-06-23.txt`: emitter aliases/types, platform identity and variants, operator country, kinematics, and location/geography.

In [ ]:
from pathlib import Path
import os, sys, json

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'combat_id_calibration').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from combat_id_calibration.graph_ingest import load_documents, extract_facts, write_facts_jsonl, populate_neo4j, _validate_neo4j_credentials

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or None
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'qwen3.5:9b')
OLLAMA_URL = os.getenv('OLLAMA_URL', 'http://localhost:11434')
FACTS_JSONL = REPO_ROOT / 'wikipedia_emitter_ontology_facts.jsonl'
print(REPO_ROOT)

## Ontology

Static sources are mined for facts that can meet dynamic CMO observations. Important predicates include `HAS_SENSOR`, `USES_RADAR`, `HAS_EMITTER`, `EMITTER_TYPE`, `PLATFORM_TYPE`, `VARIANT_OF`, `HAS_VARIANT`, `OPERATED_BY`, `OPERATOR_COUNTRY`, `TYPICAL_SPEED_KT`, `MAX_SPEED_KT`, `SERVICE_CEILING_M`, `BASED_AT`, `DEPLOYED_TO`, and `OPERATES_IN`. `graph_ingest.py` writes the auditable generic `FACT` edge and also materializes selected typed relationships/labels (`Platform`, `Sensor`, `Operator`, `Country`, `Location`) so hypothesis queries can traverse the reference graph directly.

In [ ]:
WIKIPEDIA_URLS = [
    'https://en.wikipedia.org/wiki/Mikoyan_MiG-29',
    'https://en.wikipedia.org/wiki/Mikoyan_MiG-35',
    'https://en.wikipedia.org/wiki/Zhuk_(radar)',
    'https://en.wikipedia.org/wiki/N010_Zhuk',
    'https://en.wikipedia.org/wiki/Ukrainian_Air_Force',
    'https://en.wikipedia.org/wiki/Russian_Air_Force',
    'https://en.wikipedia.org/wiki/Eurofighter_Typhoon',
    'https://en.wikipedia.org/wiki/CAPTOR-E',
]
PDF_PATHS = []

In [ ]:
# Wipe the selected Neo4j graph before rebuilding.
def wipe_neo4j(uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASSWORD, database=NEO4J_DATABASE):
    user, password = _validate_neo4j_credentials(user, password)
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        driver.verify_connectivity()
        with driver.session(**({'database': database} if database else {})) as session:
            session.run('MATCH (n) DETACH DELETE n')
    finally:
        driver.close()

wipe_neo4j()
print('Neo4j graph wiped')

In [ ]:
documents = load_documents(PDF_PATHS, WIKIPEDIA_URLS)
print(f'Loaded {len(documents)} documents')
facts = extract_facts(documents, model=OLLAMA_MODEL, ollama_url=OLLAMA_URL, max_chars=6000, overlap=500)
write_facts_jsonl(facts, FACTS_JSONL)
print(f'Wrote {len(facts)} extracted facts to {FACTS_JSONL}')

In [ ]:
populate_neo4j(facts, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
print('Populated emitter-aware reference KG')

In [ ]:
# Smoke-test emitter/platform/operator coverage for the first LuaHistory emitter.
from combat_id_calibration.hypothesis_generation import graph_hypothesis_query, emitter_aliases
from neo4j import GraphDatabase
aliases = emitter_aliases('Slot Back [N-010 Zhuk-M]')
query, params = graph_hypothesis_query(aliases, limit=20)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
try:
    with driver.session(**({'database': NEO4J_DATABASE} if NEO4J_DATABASE else {})) as session:
        rows = [dict(r) for r in session.run(query, **params)]
finally:
    driver.close()
rows